# Operating-room duration: model comparison and scheduling uncertainty

This report uses `surgery_duration.py` as the shared implementation. The target is room entry-to-exit occupancy in minutes, not incision-to-closure duration. Training is explicit; by default this notebook reads local artifacts.

The principal CCAM code and family are allowed under the professor-confirmed project assumption. Preoperative availability of diagnosis codes, procedure/diagnosis counts and anesthesia fields still needs verification; the full feature set is an experimental assumption, not a verified deployment contract.

In [ ]:
from pathlib import Path
import json
import joblib
import pandas as pd
from IPython.display import display
from surgery_duration import (run, validate_split, calibration_split, predict_schedule,
                              upper_bound_metrics, FIXED_FEATURES)
DATA_FILE = Path("resources/model_surgery_duration_dataset.parquet")
OUT = Path("results")
TRAIN_MODELS = False
TRAIN_BOUNDS = False
USE_CALENDAR = False
if TRAIN_MODELS:
    run(DATA_FILE, OUT, train_bounds=TRAIN_BOUNDS, calendar=USE_CALENDAR)
if not (OUT / "surgery_duration_model.joblib").exists():
    raise FileNotFoundError("Run surgery_duration.py --train-bounds first, or enable TRAIN_MODELS.")

## Data and evaluation protocol

Train and test retain the upstream patient-disjoint chronological partitions. Early validation is used for model selection and early stopping; later validation is reserved for calibration, excluding patients already present in early validation. The test is descriptive and never used for fitting or selection. Since historical test results have already been inspected, confirmation requires a new later cohort.

In [ ]:
df = pd.read_parquet(DATA_FILE)
train, validation, test = validate_split(df)
selection, calibration = calibration_split(validation)
display(pd.DataFrame({"partition": ["train", "selection", "calibration", "test"],
                      "cases": [len(train), len(selection), len(calibration), len(test)]}))
selection_info = json.loads((OUT / "selection.json").read_text())
print(json.dumps(selection_info, indent=2))
if "artifact_version" not in joblib.load(OUT / "surgery_duration_model.joblib"):
    print("Historical artifact: rerun the CLI to obtain the current protocol and metadata.")

## Baselines and point-model comparison

The benchmark includes nine training-only median references and multiple ML families, including weighted CatBoost as a tail-sensitive candidate. Selection is explicit: validation MAE within 0.2 minute of the best, then fastest training. Tail metrics are reported for every candidate but do not silently change this average-error selection rule. A different operational selection criterion requires a separate declared experiment.

By default, models exclude calendar features so durations remain valid when the optimizer changes dates. Calendar experiments require recalculation for each proposed date and are rejected by the fixed-duration adapter unless explicitly overridden.

In [ ]:
comparison = pd.read_csv(OUT / "surgery_duration_model_comparison.csv")
display(comparison.round(3))
print("Saved selected model:", selection_info["selected"])
validation_results = pd.read_csv(OUT / "validation_comparison.csv")
display(validation_results.sort_values("MAE_min").round(3))

## Error diagnostics

Tail thresholds come from training P90, never from the test distribution. Signed error is prediction minus actual; a negative value denotes underprediction. Compare average accuracy and severe underprediction before drawing scheduling conclusions.

In [ ]:
import matplotlib.pyplot as plt
if (OUT / "duration_band_errors.csv").exists():
    display(pd.read_csv(OUT / "duration_band_errors.csv").round(3))
if (OUT / "feature_importance.csv").exists():
    importance = pd.read_csv(OUT / "feature_importance.csv")
    display(importance.round(3))
    importance.head(10).plot.barh(x="feature", y="MAE_increase_min", legend=False)
    plt.xlabel("Validation MAE increase after permutation (min)")
    display(plt.gcf())
    plt.close()

## P80/P95 bounds and the cost of protection

Optional unweighted CatBoost quantile models are calibrated on the reserved later validation subset. Bounds are ordered upward after calibration. Coverage is empirical: temporal drift and nonexchangeable cases prevent an unconditional guarantee. A patient's P95 bound is not a 95% guarantee for a whole vacation. `uncertainty_width_min` is P95 minus point, not a two-sided prediction interval.

Train bounds with `surgery_duration.py --train-bounds`, or set both TRAIN_MODELS and TRAIN_BOUNDS above. Merely setting TRAIN_BOUNDS does not retrain or change artifacts.

In [ ]:
predictions = predict_schedule(test, OUT / "surgery_duration_model.joblib")
# Keep row-level predictions local; display aggregate summaries only.
if "duration_p95_min" in predictions:
    display(pd.DataFrame([dict(bound=label, **upper_bound_metrics(test["target_surgery_duration_min"],
        predictions[f"duration_{label}_min"])) for label in ("point", "p80", "p95")]).round(3))
    extra = pd.DataFrame({label: predictions[f"duration_{label}_min"] - predictions.duration_point_min
                          for label in ("p80", "p95")})
    display(extra.describe(percentiles=[.5, .9, .95]).round(3))
    buffers = predictions[["duration_point_min", "duration_p80_min", "duration_p95_min"]].subtract(test["target_surgery_duration_min"], axis=0)
    display(pd.DataFrame({"mean_buffer_min": buffers.mean(), "median_buffer_min": buffers.median(),
                          "mean_absolute_error_min": buffers.abs().mean()}).round(3))
else:
    print("Point predictions available; bounds have not been trained in this artifact.")

## Reusable inference and optimizer integration

`predict_schedule(frame, artifact, raw=False)` accepts prepared modeling features and returns point and available P80/P95 durations. With `raw=True`, it accepts EDA-schema preoperative rows and applies the saved training-only category mapping and shared feature engineering. The preprocessing notebook must first save that mapping. No realized room timestamps are used.

`apply_predicted_durations(patients, features, artifact, risk="point", raw=False)` validates aligned unique indexes and finite positive durations, and returns the optimizer patient schema with the selected duration. It supports `risk="p80"` or `"p95"` when bounds are saved. Scheduling benefit still needs evaluation using room utilization, overruns and bed constraints; lower MAE alone is insufficient.